# Building a neural network from scratch

In this module, you will build a neural network which will learn a very simple function, a polynomial function of the sort: 

\begin{equation}
y = A + B \cdot x + C \cdot x^2 + D \cdot x^3
\end{equation}

You can choose the value of coefficients for this exercise. Later on, we will see what happens when we add noise. 

But first, let us plot and see your function! 

In [ ]:
import numpy as np 
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from src.utils import degree_3_polynomial
A = 1 
B = 2 
C = -0.5 
D = 0.04 
x_true = np.linspace(0, 10, 1000)
y_true = degree_3_polynomial(x_true, coeffs=[A, B, C, D], noise_std=0.2) # gaussian noise with stdev added

plt.plot(x_true, y_true, color="black")
plt.xlabel('X')
plt.ylabel('Y')

# Learning from data
The function that you plotted above is, of course, arbitrary: you chose the coefficients of the polynomial yourself. The choice of using a polynomial is also somewhat arbitrary—we use it here because it is simple and easy to understand. In real-world applications, however, such a function often represents a meaningful signal. A one-dimensional function could correspond to the output of a sensor, for example a temperature measurement over time, the concentration of a biomolecule in an experiment, or the growth of a cell population. Over a limited range, such signals can often be approximated by simple functions like polynomials.

In practice, you would not have access to the full continuous function. Instead, you would only observe a finite number of discrete data points sampled from it. From these samples, you aim to reconstruct, or learn, the underlying relationship. This process is known as _machine learning_.

Below, you can see a set of sampled data points from the function you generated earlier. The red points represent the observed samples, while the dashed line indicates the underlying (true) function that generated them.

In [ ]:
sample_size = 20
sampling_indices = np.random.choice(len(x_true), size=sample_size, replace=False)
x_sample = x_true[sampling_indices]
y_sample = y_true[sampling_indices] 

plt.plot(x_true, y_true, color="black", label="True Function", linestyle='--', alpha=0.2)
plt.scatter(x_sample, y_sample, color="red", label="Sampled Data")
plt.legend()

# show sampled points as a table 

data = {'X': x_sample, 'Y': y_sample}
df = pd.DataFrame(data)
print(df.head())

## Fit with a polynomial function

A direct approach is to estimate the coefficients of a polynomial function using the least squares method. This method finds the polynomial that best fits the observed data by minimizing the difference between the predicted values and the data points.

In practice, we can use standard functions from the NumPy library to perform this fitting. For example, the function `polyfit` allows you to compute the coefficients of a polynomial of a chosen degree that best fits your data.

You can find more details in the official [documentation](https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html). 


In [ ]:
# Fitting a polynomial of degree 3 to the data
coefficients = np.polyfit(x_sample, y_sample, deg=3) 
# to create a polynomial function from the coefficients
fitted_polynomial = np.poly1d(coefficients) 
# find the fit function using true x values
y_fitted = fitted_polynomial(x_true)
plt.plot(x_true, y_true, color="black", label="True Function", linestyle='--', alpha=0.2) 
plt.plot(x_true, y_fitted, color="blue", label="Fitted Polynomial")
plt.scatter(x_sample, y_sample, color="red", label="Sampled Data")
plt.xlabel('X')
plt.ylabel('Y')
plt.ylim(-1, 12)
plt.legend()


# Learning through neurons

In this module, we will see how the same function can be learned using a different framework—one loosely inspired by the human brain. A (biological) neuron is a cell that receives input signals, processes them, and produces an output signal. While artificial neurons are inspired by this idea, the similarity is quite limited. In practice, an artificial neuron is simply a mathematical function that maps inputs to an output.

Biological neurons are often simplified as having a binary behavior: they either “fire” or they do not. This behavior can be approximated by a smooth transition using a sigmoid function, where the output gradually changes between 0 and 1.

In artificial neural networks, however, a neuron can be modeled using many different mathematical functions. These are typically non-linear functions, which allow the network to learn complex patterns. In this exercise, we will focus on three commonly used activation functions: _ReLU_, _sigmoid_, and _tanh_.

The ReLU function is defined as:
\begin{equation}
\tag{2a}
f(x) = \begin{cases} 
      0 & x < 0 \\
      x & x \geq 0 
   \end{cases}
\end{equation}

The sigmoid function is defined as:
\begin{equation}
\tag{2b}
f(x) = \frac{1}{1 + e^{-x}}
\end{equation}

The tanh function is defined as:
\begin{equation}
\tag{2c}
f(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}
\end{equation}

Can you implement the ReLU, sigmoid and tanh functions in Python? 


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
def activation_function(z, function_type="sigmoid"):
    if function_type == "linear":
        return z # a linear activation function simply returns the same input
    if function_type == "sigmoid":
        return 

    elif function_type == "relu":
       return 

    elif function_type == "tanh":
       return 
         
    else:
        raise ValueError("Unsupported activation function.")



Test your implementation by plotting the functions in the next code cell

In [ ]:
# plot sigmoid,  relu, and tanh functions
z = np.linspace(-10, 10, 400)
sigmoid_values = activation_function(z, function_type="sigmoid")
relu_values = activation_function(z, function_type="relu")
tanh_values = activation_function(z, function_type="tanh")
fig, ax = plt.subplots(1, 3, figsize=(6, 2))
ax[0].plot(z, sigmoid_values)
ax[0].set_title("Sigmoid")
ax[1].plot(z, relu_values)
ax[1].set_title("ReLU")
ax[2].plot(z, tanh_values)
ax[2].set_title("Tanh")

plt.tight_layout()


Learning with a neural network essentially involves finding a combination of activation functions and parameters that best approximates the dataset on which the model is trained. As we will see later in this module, this approach is powerful because it allows us to flexibly approximate a wide range of functions by combining simple non-linear transformations.


### The behaviour of neurons
To build an intuitive understanding of how neurons “learn”, we will first explore how changing the parameters of an activation function affects its behavior.

In the following code cell, you can control two parameters: the __slope__ and the __activation point__. By adjusting these, you can observe how the shape of the activation function changes. In neural networks, the slope is often called the weight. The activation point is the value of x at which the neuron changes its response most strongly. For example, for sigmoid and tanh this is the center of the transition, and for ReLU it is the point where the output starts increasing.

In the standard neural network form, a neuron is written as $f(wx+b)$, where $w$ is the weight and $b$ is the bias. In this widget, we use the activation point instead of the bias because it is more intuitive to control visually. The bias is related to the slope and activation point through $b = −w⋅x_a$, where $x_{a}$ is the activation point. This means that the activation point depends on both the weight and the bias.

You can also modify the input value, which represents the value of $x$ fed into the activation function. On the right, you will see how these parameter changes are reflected in the output of the neuron.

In [ ]:
# Add an interact to visualise effect of parameters on the neuron output
# Use ReLU, PReLU, sigmoid, and tanh as activation functions
import ipywidgets as widgets
from IPython.display import display
from src.utils import update_neuron_output_using_activation_point
    
slope_slider = widgets.FloatSlider(min=-5, max=5, step=0.1, value=1, description='Slope')
activation_point_slider = widgets.FloatSlider(min=-10, max=10, step=0.1, value=0, description='Act. point')
x_slider = widgets.FloatSlider(min=-10, max=10, step=0.1, value=5, description='Input x')
activation_function_dropdown = widgets.Dropdown(options=['relu', 'sigmoid', 'tanh'], value='relu', description='Activation')
widgets.interact(\
    update_neuron_output_using_activation_point, \
    function_type=activation_function_dropdown, \
    slope=slope_slider, \
    activation_point=activation_point_slider,\
    x=x_slider
);



Next, you will use these sliders to control the shape of three “neurons”. In addition, you can adjust the weight assigned to each neuron to see how their outputs combine into a final result. Note that these weights can be either positive or negative.

Your goal is to find a set of parameters for these three neurons such that their combined output approximates the function you plotted at the beginning of this module.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from src.utils import update_layer_of_neurons_using_slope_activation_point
min_val, max_val, step = -5, 10, 0.01
weight_min, weight_max = -5, 10

# Sliders
slope_1 = widgets.FloatSlider(min=min_val, max=max_val, step=step, value=0, description='Slope 1')
slope_2 = widgets.FloatSlider(min=min_val, max=max_val, step=step, value=0, description='Slope 2')
slope_3 = widgets.FloatSlider(min=min_val, max=max_val, step=step, value=0, description='Slope 3')

ap_1 = widgets.FloatSlider(min=min_val, max=max_val, step=step, value=0, description='Act. Pt 1')
ap_2 = widgets.FloatSlider(min=min_val, max=max_val, step=step, value=0, description='Act. Pt 2')
ap_3 = widgets.FloatSlider(min=min_val, max=max_val, step=step, value=0, description='Act. Pt 3')

weight_1 = widgets.FloatSlider(min=weight_min, max=weight_max, step=step, value=1, description='Weight 1')
weight_2 = widgets.FloatSlider(min=weight_min, max=weight_max, step=step, value=1, description='Weight 2')
weight_3 = widgets.FloatSlider(min=weight_min, max=weight_max, step=step, value=1, description='Weight 3')

output_bias = widgets.FloatSlider(min=min_val, max=max_val, step=step, value=0, description='Output Bias', 
                                      orientation='vertical')
function_dropdown = widgets.Dropdown(options=['relu', 'sigmoid', 'tanh'], value='relu', description='Activation')

sample_data = (x_sample, y_sample, y_true)
# interactive_output wires up callbacks but does NOT render the widgets itself
out = widgets.interactive_output(
    update_layer_of_neurons_using_slope_activation_point,
    {
        'slope_1': slope_1, 'slope_2': slope_2, 'slope_3': slope_3,
        'activation_point_1': ap_1, 'activation_point_2': ap_2, 'activation_point_3': ap_3,
        'weight_1': weight_1, 'weight_2': weight_2, 'weight_3': weight_3,
        'output_bias': output_bias, "function_type":function_dropdown,
        'sample_data': widgets.fixed(sample_data)
    }
)

# Group sliders by neuron so each column corresponds to one neuron
neuron_1_controls = widgets.VBox([slope_1, ap_1, weight_1])
neuron_2_controls = widgets.VBox([slope_2, ap_2, weight_2])
neuron_3_controls = widgets.VBox([slope_3, ap_3, weight_3])
all_controls = widgets.HBox([
    widgets.VBox([neuron_1_controls, neuron_2_controls, neuron_3_controls, function_dropdown]),
    output_bias
])

# Sliders on the left, plot on the right
display(widgets.HBox([all_controls, out]))

Could you obtain a good fit? Were you able to reduce the error to be less than $10^{-2}$, or even less than $10^{-3}$?

## Think! 
- Why do you think three neurons were sufficient for this curve? 
- What would happen if you had only two neurons? 
- What if you had a million neurons?


# Artificial Neural Networks

In biological neurons, the connection strengths between cells are modulated by complex biochemical processes. In artificial neural networks, these connection strengths are represented by numerical values called __weights__. A weight can be either positive or negative, and it determines how strongly an input influences the output of the neuron.

Each neuron computes a weighted sum (a linear combination) of its inputs and then applies a non-linear activation function before passing the result to the next layer.

Mathematically, the output of a single neuron can be expressed as: <br><br>


\begin{equation}
\tag{2}
y = f\left(\sum_{i=1}^{n} w_i x_i + b\right)
\end{equation}

where:
- $x_i$ are the inputs to the neuron
- $w_i$ are the weights associated with each input, similar to the slope you explored earlier
- $b$ is the bias term, which shifts the activation function and, together with the weights, determines the activation point
- $f$ is the activation function (e.g., ReLU, sigmoid)
- $y$ is the output of the neuron

To understand how artificial neural networks can “learn” to approximate data, you will now build a simple feedforward neural network with one hidden layer. The hidden layer will consist of three neurons, and the output layer will consist of a single neuron that produces the final output.

This setup is similar to what you explored earlier with three neurons, but now we introduce a more formal structure and notation.

We will begin by implementing a __forward pass__, where the input data is passed through the network to produce an output. After that, you will implement __backpropagation__, which adjusts the parameters (weights and biases) based on the error between the predicted output and the true output.

### Forward pass

In the first task, you will write a function that computes the output of a neuron, as defined in Equation (2). This function should take as input the data, the weights, the bias, and the activation function, and return the output of the neuron.

In [ ]:
def neuron(x, weight, bias, function_type="relu"):
    # compute the linear transformation
    z = 
    # apply the activation function to z
    y = 
    return y

In [ ]:
x = x_sample[0] # Take the first sample input for demonstration
y = y_sample[0] # Take the corresponding true output for demonstration
# What should be the shape of weights and bias for a single neuron which takes 
# x as input and produces a single output?
# To keep it simple, give values between -1 and 1 for weights and bias.

weights = np.array([]) # Give a random value for weight. 
bias =  # Give a random value for bias

# compute the output of the neuron using ReLU activation function
output = neuron(x, weights, bias, function_type="relu")

# print the input, output, and true output
print("Input to the neuron:", x)
print("Output of the neuron:", output)
print("True output:", y)

Great! Now suppose that we have a layer of three neurons, each with its own set of weights and biases. What would be the output of this layer? 


In [ ]:
def layer_of_neurons(x, list_of_neurons, function_type="relu"):
    outputs = []
    for neuron_params in list_of_neurons:
        weight, bias = neuron_params
        output = neuron(x, weight, bias, function_type) # write your function here
        outputs.append(output)
    return np.array(outputs).squeeze() # return the outputs as a numpy array


Let us define three “neurons” that share the same activation function but have different weights and biases. We will then combine the outputs of these three neurons to obtain the final output.

Each neuron is represented in Python as a list with the following structure:

`neuron = [weight_vector, bias]`

Where:
where:

- `weight_vector` is a NumPy array of shape `(n_inputs,)` containing the weights for each input
- `bias` is a scalar (or a NumPy array of shape `(1,)`) representing the bias term of the neuron


In [ ]:
w_1_1, b_1_1 =  # weight, bias from input to neuron 1 in layer 1. 
w_2_1, b_2_1 =  # weight, bias from input to neuron 2 in layer 1
w_3_1, b_3_1 =  # weight, bias from input to neuron 3 in layer 1

neuron_1 = (w_1_1, b_1_1)
neuron_2 = (w_2_1, b_2_1)
neuron_3 = (w_3_1, b_3_1)

In [ ]:
hidden_layer_output = layer_of_neurons(
    x=x, \
    list_of_neurons=[neuron_1, neuron_2, neuron_3], \
    function_type="relu",\
)
print(f"Input to the layer: {x}")
print(f"Output of the layer: {hidden_layer_output}")

Does the shape of the output of the hidden layer make sense to you?

Now, combine the outputs of the hidden layer to obtain the final output. The final output is a __linear combination__ of the outputs of the hidden layer neurons, weighted by the weights of the output neuron.

In many cases, the output neuron does not use an activation function and is therefore kept __linear__. This means it simply computes a weighted sum of its inputs.

In [ ]:
w_1_2, w_2_2, w_3_2 =  # weights from hidden layer to output neuron
b_1_2 = # bias for output neuron 

# define the output neuron using the weights and bias from hidden layer to output neuron
# think carefull about the shape of the weight vector and bias for the output neuron
output_neuron = () 

final_output = layer_of_neurons(
    x=hidden_layer_output, \
    list_of_neurons=[output_neuron], \
    function_type="relu",
)

print(f"Output of the network: {final_output}")
print(f"True output: {y}")


### Evaluating the network output
How does the output of your network compare to the true function? Is it close, or does it deviate significantly?

In the next section, you will implement a method to **quantify this difference**.

---

### Measuring error

The difference between the predicted output and the true output can be quantified using a **loss function**. A commonly used loss function for regression problems is the **Mean Squared Error (MSE)**, defined as: <br><br>

\begin{equation} 
\tag{3}
MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
\end{equation}

where:
- $y_i$ is the true output for the $i$-th data point  
- $\hat{y}_i$ is the predicted output for the $i$-th data point  
- $n$ is the number of data points  

In our case, we may evaluate one data point at a time (so $n = 1$), but in general this formula applies to any number of data points.



In [ ]:
def convert_scalar_to_array(*value):
    converted_values = []
    for val in value:
        if np.isscalar(val):
            converted_values.append(np.array([val]))
        else:
            converted_values.append(val)
    return converted_values

def mse_loss(y_true, y_pred):
    # check if y_true or y_pred are scalars
    y_true, y_pred = convert_scalar_to_array(y_true, y_pred)
    # write your code to compute mse loss using Equation 2
    n =  
    loss_value = 
    
    return loss_value

In [ ]:
mse_loss_value = mse_loss(y, final_output)
print(f"MSE Loss: {mse_loss_value}")

# Backpropagation

Now that we have computed the loss between the predicted output and the true output, the next step is to understand how this information is used to adjust the parameters of the network. This is done through a process called **backpropagation**.

In backpropagation, the error is first computed at the final output of the network and then propagated backward through the layers. The overall flow of information in a neural network can be summarized as:

- **Forward pass**: input data flows through the network to produce an output  
- **Backward pass**: the error signal is propagated back through the network to update the parameters 

<br>

---

### Output error signal

We begin by computing how the loss changes with respect to the final output. This is given by the derivative of the loss function with respect to the predicted output.

For the Mean Squared Error (MSE) loss function, and for the case $n = 1$, we have:


\begin{equation*}
\tag{4}
MSE = \frac{1}{n} (y_i - \hat{y}_i)^2 = (y - \hat{y})^2
\end{equation*}

Taking the derivative with respect to the predicted output $\hat{y}$, we obtain:

\begin{equation}
\tag{5}
\frac{\partial MSE}{\partial \hat{y}_{\text{final}}} = \delta_{\text{out}} = -\frac{2}{n} (y_i - \hat{y}_i) = -2 (y - \hat{y})
\end{equation}

The term $\delta_{\text{out}}$ is called the **output error signal**. It quantifies how sensitive the loss is to changes in the final output of the network, and serves as the starting point for propagating the error backward through the network.

In [1]:
# Compute the loss 
def mse_gradient_output_neuron(y_true, y_pred):
    # check if y_true or y_pred are scalars
    y_true, y_pred = convert_scalar_to_array(y_true, y_pred)

    # write your code to compute the gradient of MSE loss 
    n = 
    gradient =  # write your MSE gradient function here
    return gradient.squeeze() 

SyntaxError: invalid syntax (14592684.py, line 7)

In [ ]:
output_error_signal = mse_gradient_output_neuron(y, final_output)
print(f"MSE Gradient: {output_error_signal}")

What does the gradient actually tell us? To build an intuitive understanding, let us visualise the loss function and plot the gradient on top of it.

In [ ]:
from src.utils import plot_loss_landscape_with_state, print_summary_of_network

In [ ]:
states = {
    'Current prediction' : {
        'prediction': final_output,
        'loss': mse_loss_value,
        'condition': 'Current prediction',
        'output_error_signal' : output_error_signal
    },
}

plot_loss_landscape_with_state(
    loss_fn=mse_loss, output_vector=y,
    states=states, tangent=states['Current prediction'], window_size=5 
)

What the output error signal tells us is how steep the loss function is at the current predicted point. This information is essential for determining the direction in which we should adjust the parameters to reduce the error.

Each parameter must be updated so that the final prediction moves toward a minimum in this loss landscape. This process of iteratively updating the parameters to reduce the error is called **gradient descent**.

The update rule for gradient descent is given by:

\begin{equation}
\tag{6}
\begin{aligned}
w_i &\leftarrow w_i - \eta \cdot \frac{\partial MSE}{\partial w_i} \\
b_i &\leftarrow b_i - \eta \cdot \frac{\partial MSE}{\partial b_i}
\end{aligned}
\end{equation}

where $\eta$ is the **learning rate**, which determines how much we update the parameters in each iteration. Here, $w_i$ and $b_i$ are the weights and biases of the neurons in the network, and $ \frac{\partial MSE}{\partial w_i} $ and $ \frac{\partial MSE}{\partial b_i} $ are the gradients of the loss function with respect to these parameters.

Now, write a function to update the weights and biases of the output neuron using the gradients and the learning rate, as given in Equation (6).

In [ ]:
def update_weights(neuron, dL_dw, dL_db, learning_rate=0.01):
    weight, bias = neuron
    # compute the weight update using the learning rate and the gradient
    weight_update = 
    # compute the bias update using the learning rate and the gradient
    bias_update = 

    # update weights and bias
    new_weight = 
    new_bias = 
    return new_weight, new_bias

We now need to compute these gradients from the output error signal $\delta_{\text{out}}$ that we defined above.

To do this, we recursively apply the **chain rule** from calculus to propagate the gradients through the network, layer by layer. Let us begin with the output layer.

The forward pass of the output neuron is given by:

\begin{equation}
\tag{7}
\hat{y}_{final} = w_1^{(2)} y_1^{(1)} + w_2^{(2)} y_2^{(1)} + w_3^{(2)} y_3^{(1)} + b_1^{(2)}
\end{equation}

where $y_i^{(1)}$, for $i \in \{1, 2, 3\}$, are the outputs of the three neurons in the first (hidden) layer, and $w_i^{(2)}$ are the weights of the output neuron. 

Note that the output neuron does not have an activation function, so the output is simply a linear combination of the hidden layer outputs. Later, we will compute the gradients for the hidden layer neurons, which do include an activation function.

The required partial derivatives are:

\begin{equation}
\tag{7a}
\frac{\partial \hat{y}_{final}}{\partial w_1^{(2)}} = y_1^{(1)}
\end{equation}

\begin{equation}
\tag{7b}
\frac{\partial \hat{y}_{final}}{\partial w_2^{(2)}} = y_2^{(1)}
\end{equation}

\begin{equation}
\tag{7c}
\frac{\partial \hat{y}_{final}}{\partial w_3^{(2)}} = y_3^{(1)}
\end{equation}

\begin{equation}
\tag{7d}
\frac{\partial \hat{y}_{final}}{\partial b_1^{(2)}} = 1
\end{equation}

In [ ]:
# Plug in the right variables to get the correct gradients. For instance, 
y_final =  # final output of the network is called y_final

y_1_1 = # what should go here?
y_2_1 = # what should go here?
y_3_1 = # what should go here?

dy_final_dw_1_2 =  # write the correct variable here
dy_final_dw_2_2 =  # write the correct variable here
dy_final_dw_3_2 =  # write the correct variable here
dy_final_db_1_2 = 1 # the derivative of the output with respect to the bias is always 1

Now, we can use the **chain rule** to calculate the gradient of the loss function $MSE$ with respect to the weights and bias of the output neuron. For example, for the weight $w_1^{(2)}$, we have: 
<br><br>


\begin{equation}
\tag{8a}
\frac{\partial MSE}{\partial w_1^{(2)}} = \frac{\partial MSE}{\partial \hat{y}_{final}} \cdot \frac{\partial \hat{y}_{final}}{\partial w_1^{(2)}} = \delta_{out} \cdot y_1^{(1)}
\end{equation}

\begin{equation}
\tag{8b}
\frac{\partial MSE}{\partial w_2^{(2)}} = \frac{\partial MSE}{\partial \hat{y}_{final}} \cdot \frac{\partial \hat{y}_{final}}{\partial w_2^{(2)}} = \delta_{out} \cdot y_2^{(1)}
\end{equation}

\begin{equation}
\tag{8c}
\frac{\partial MSE}{\partial w_3^{(2)}} = \frac{\partial MSE}{\partial \hat{y}_{final}} \cdot \frac{\partial \hat{y}_{final}}{\partial w_3^{(2)}} = \delta_{out} \cdot y_3^{(1)}
\end{equation}

\begin{equation}
\tag{8d}
\frac{\partial MSE}{\partial b_1^{(2)}} = \frac{\partial MSE}{\partial \hat{y}_{final}} \cdot \frac{\partial \hat{y}_{final}}{\partial b_1^{(2)}} = \delta_{out}
\end{equation}

<br><br>

Now, write code to compute the gradients of the weights and bias of the output neuron using Equation (8).



In [ ]:
# write the correct expression for the gradient of the loss with respect to weight 1 in layer 2
dL_dw_1_2 = output_error_signal * dy_final_dw_1_2 
# write the correct expression for the gradient of the loss with respect to weight 2 in layer 2
dL_dw_2_2 = 
# write the correct expression for the gradient of the loss with respect to weight 3 in layer 2
dL_dw_3_2 = 
# write the correct expression for the gradient of the loss with respect to bias in layer 2
dL_db_1_2 = 

Now, update the weights and bias of the output neuron using the gradients that we calculated. After updating, calculate the new loss and see if it has decreased.

In [ ]:
# update the weights and bias for neuron 1 in layer 2
# we have filled this in for you. Make sure the variable names in this function
# match the variable names you have used to compute the gradients above.

learning_rate = 0.01 
updated_output_neuron = update_weights(\
    neuron=output_neuron, \
    dL_dw=np.array([dL_dw_1_2, dL_dw_2_2, dL_dw_3_2]), \
    dL_db=dL_db_1_2, \
    learning_rate=learning_rate, \
)

print(f"Updated output neuron weights: {updated_output_neuron[0]}, bias: {updated_output_neuron[1]}")
print(f"Original output neuron weights: {output_neuron[0]}, bias: {output_neuron[1]}")

Now let us calculate the gradients for the weights and biases of the three neurons in the previous layer. The forward pass for these neurons is given by:

\begin{equation*}
\begin{aligned}
y_1^{(1)} &= f(w_1^{(1)} x + b_1^{(1)}) = f(z_1^{(1)}) \\
y_2^{(1)} &= f(w_2^{(1)} x + b_2^{(1)}) = f(z_2^{(1)}) \\
y_3^{(1)} &= f(w_3^{(1)} x + b_3^{(1)}) = f(z_3^{(1)})
\end{aligned}
\end{equation*}

where $f$ is the activation function (ReLU in our case). To keep the notation simple, we will show the chain rule only for the first neuron, but the same reasoning applies to the other two neurons as well.

The derivatives of $y_1^{(1)}$ with respect to its parameters are given by:

\begin{equation*}
\begin{aligned}
\frac{\partial y_1^{(1)}}{\partial w_1^{(1)}} &= \frac{\partial y_1^{(1)}}{\partial z_1^{(1)}} \cdot \frac{\partial z_1^{(1)}}{\partial w_1^{(1)}} \\
\frac{\partial y_1^{(1)}}{\partial b_1^{(1)}} &= \frac{\partial y_1^{(1)}}{\partial z_1^{(1)}} \cdot \frac{\partial z_1^{(1)}}{\partial b_1^{(1)}}
\end{aligned}
\end{equation*}

We therefore need the derivative of the activation function with respect to its input $z_1^{(1)}$. We denote this derivative by $f'$. For ReLU, the derivative is:

\begin{equation*}
\frac{\partial y_i^{(1)}}{\partial z_i^{(1)}} = f'(z_i^{(1)}) =
\begin{cases}
1 & \text{if } z_i^{(1)} > 0 \\
0 & \text{if } z_i^{(1)} \leq 0
\end{cases}
\end{equation*}

Note that the derivative of ReLU is not formally defined at $z=0$. In practice, we simply set it to $0$ by convention, which is sufficient for this exercise.

Using this, we obtain:

\begin{equation*}
\begin{aligned}
\frac{\partial y_1^{(1)}}{\partial w_1^{(1)}} &= \frac{\partial y_1^{(1)}}{\partial z_1^{(1)}} \cdot \frac{\partial z_1^{(1)}}{\partial w_1^{(1)}} = f'(z_1^{(1)}) \cdot x \\
\frac{\partial y_1^{(1)}}{\partial b_1^{(1)}} &= \frac{\partial y_1^{(1)}}{\partial z_1^{(1)}} \cdot \frac{\partial z_1^{(1)}}{\partial b_1^{(1)}} = f'(z_1^{(1)})
\end{aligned}
\end{equation*}

Next, we need the derivative of the final output $\hat{y}_{final}$ with respect to the output of the first hidden neuron $y_1^{(1)}$. From the forward pass of the output neuron in Equation (5a), we have:

\begin{equation*}
\frac{\partial \hat{y}_{final}}{\partial y_1^{(1)}} = w_1^{(2)}
\end{equation*}

We can now apply the chain rule to calculate the gradient of the loss function with respect to the weight of the first hidden neuron:


\begin{aligned}
\tag{9a}
\frac{\partial MSE}{\partial w_1^{(1)}} &= \frac{\partial MSE}{\partial \hat{y}_{final}} \cdot \frac{\partial \hat{y}_{final}}{\partial y_1^{(1)}} \cdot \frac{\partial y_1^{(1)}}{\partial w_1^{(1)}} \\
&= \delta_{out} \cdot w_1^{(2)} \cdot f'(z_1^{(1)}) \cdot x
\end{aligned}


Similarly, for the bias of the first hidden neuron:


\begin{aligned}
\tag{9b}
\frac{\partial MSE}{\partial b_1^{(1)}} &= \frac{\partial MSE}{\partial \hat{y}_{final}} \cdot \frac{\partial \hat{y}_{final}}{\partial y_1^{(1)}} \cdot \frac{\partial y_1^{(1)}}{\partial b_1^{(1)}} \\
&= \delta_{out} \cdot w_1^{(2)} \cdot f'(z_1^{(1)})
\end{aligned}


Now, write code to calculate the gradients for the weight and bias of the first hidden neuron using Equations (7a) and (7b). You can then repeat the same process for the other two hidden neurons.

In the next cell, we have already loaded functions to compute the derivatives of the activation functions. You can use these functions to calculate the derivatives for the hidden layer neurons.

In [ ]:
from src.utils import relu_derivative, sigmoid_derivative, tanh_derivative

In [ ]:
# write the correct expression for the gradient of the loss with respect to weight 1 in layer 1
dL_dw_1_1 = 
dL_db_1_1 = 
print(f"Gradient for weight 1 in layer 1: {dL_dw_1_1}, Gradient for bias in layer 1: {dL_db_1_1}")


In [ ]:
# write the correct expression for the gradient of the loss with respect to weight 2 in layer 1
dL_dw_2_1 = 
dL_db_2_1 = 

In [ ]:
# write the correct expression for the gradient of the loss with respect to weight 3 in layer 1
dL_dw_3_1 = 
dL_db_3_1 = 

In [ ]:
# update the weights and bias for neuron 1 in layer 1
updated_neuron_1 = update_weights(\
    neuron=neuron_1, \
    dL_dw=dL_dw_1_1, \
    dL_db=dL_db_1_1, \
    learning_rate=learning_rate, \
)
# update the weights and bias for neuron 2 in layer 1
updated_neuron_2 = update_weights(\
    neuron=neuron_2, \
    dL_dw=dL_dw_2_1, \
    dL_db=dL_db_2_1, \
    learning_rate=learning_rate, \
)
# update the weights and bias for neuron 3 in layer 1
updated_neuron_3 = update_weights(\
    neuron=neuron_3, \
    dL_dw=dL_dw_3_1, \
    dL_db=dL_db_3_1, \
    learning_rate=learning_rate, \
)


In [ ]:
# print the updated weights and biases for all neurons in the network
print(f"Updated neuron 1 weight: {updated_neuron_1[0]}, bias: {updated_neuron_1[1]}")
print(f"Updated neuron 2 weight: {updated_neuron_2[0]}, bias: {updated_neuron_2[1]}")
print(f"Updated neuron 3 weight: {updated_neuron_3[0]}, bias: {updated_neuron_3[1]}")


In [ ]:
updated_layer_1_neurons = [updated_neuron_1, updated_neuron_2, updated_neuron_3]
updated_layer_2_neurons = [updated_output_neuron]


In [ ]:
# What is the new prediction of the network after the weight updates? 
# Create a function called forward_pass that takes the input x
# and two layers of neurons and returns the final output of the network
def forward_pass(x, layer_1_neurons, layer_2_neurons, function_type="relu"):
    hidden_layer_output = layer_of_neurons() # fill in the parameters
    final_output = layer_of_neurons() # fill in the parameters
    return final_output


In [ ]:
new_prediction = forward_pass(\
    x=x, \
    layer_1_neurons=updated_layer_1_neurons, \
    layer_2_neurons=updated_layer_2_neurons, \
    function_type="relu"
)
print(f"New prediction after weight updates: {new_prediction}")
print(f"Earlier prediction before weight updates: {final_output}")
print(f"True output: {y}")

Is the network output moving in the right direction? Run the next code cell to visualise how it changes.

In [ ]:
updated_loss = mse_loss(y, new_prediction)
updated_output_error_signal = mse_gradient_output_neuron(y, new_prediction)

states.update({
    'Updated prediction' : {
        'prediction': new_prediction,
        'loss': updated_loss,
        'condition': 'After one weight update step',
        'output_error_signal' : updated_output_error_signal
    },
})
plot_loss_landscape_with_state(
    loss_fn=mse_loss, output_vector=y,
    states=states, tangent=states['Updated prediction'], window_size=5
)

Now, write a loop which repeats this process of forward pass multiple times. In each iteration, calculate the loss and update the parameters using backpropagation. You should see the loss decreasing over iterations as the network learns to approximate the function.


In [ ]:
n_iter = 50
x_train = x_sample[0] # again we will use the first sample input for training
y_train = y_sample[0] # corresponding true output for training
learning_rate = 0.1
states_training = {}
for i in range(n_iter):
    # Forward pass
    if i == 0:
        # initialise random weights and biases for the first iteration
        w_1_1, b_1_1 = #   weight, bias from input to neuron 1 in layer 1
        w_2_1, b_2_1 = # 
        w_3_1, b_3_1 = # 
        w_1_2, w_2_2, w_3_2 = # 
        b_1_2 = # 

    neuron_1 = (w_1_1, b_1_1)
    neuron_2 = (w_2_1, b_2_1)
    neuron_3 = (w_3_1, b_3_1)

    # Create the output neuron
    output_neuron = ()

    # Define layer 1 neurons 
    layer_1_neurons = [] 
    layer_2_neurons = []
    hidden_layer_1 = layer_of_neurons(x_train, layer_1_neurons, function_type="relu")
    final_output = layer_of_neurons(hidden_layer_1, layer_2_neurons, function_type="relu")
    
    loss_value = mse_loss(
        y_true=, 
        y_pred= # fill in 
    )
    output_error_signal = mse_gradient_output_neuron(
        y_true=, 
        y_pred=, # fill in
    )

    # Compute gradients for output layer
    dL_dw_1_2 = 
    dL_dw_2_2 = 
    dL_dw_3_2 = 
    dL_db_1_2 = 

    # Compute gradients for hidden layer
    dL_dw_1_1 = 
    dL_db_1_1 = 
    dL_dw_2_1 = 
    dL_db_2_1 = 
    dL_dw_3_1 = 
    dL_db_3_1 = 

    # Update weights and biases for output layer
    output_neuron = update_weights(\
        neuron=output_neuron, \
        dL_dw=np.array([dL_dw_1_2, dL_dw_2_2, dL_dw_3_2]), \
        dL_db=dL_db_1_2, \
        learning_rate=learning_rate, \
    )

    # Update weights and biases for hidden layer
    neuron_1 = update_weights(\
        neuron=neuron_1, \
        dL_dw=dL_dw_1_1, \
        dL_db=dL_db_1_1, \
        learning_rate=learning_rate, \
    )
    neuron_2 = update_weights(\
        neuron=neuron_2, \
        dL_dw=dL_dw_2_1, \
        dL_db=dL_db_2_1, \
        learning_rate=learning_rate, \
    )
    neuron_3 = update_weights(\
        neuron=neuron_3, \
        dL_dw=dL_dw_3_1, \
        dL_db=dL_db_3_1, \
        learning_rate=learning_rate, \
    )

    # update the weights and biases for the next iteration
    w_1_1, b_1_1 = neuron_1
    w_2_1, b_2_1 = neuron_2
    w_3_1, b_3_1 = neuron_3
    w_1_2, w_2_2, w_3_2 = output_neuron[0]
    b_1_2 = output_neuron[1]
    
    # Leave a record of the state of the network at each iteration for visualisation later
    states_training["iteration_{}".format(i+1)] = {
        'prediction': final_output,
        'loss': loss_value,
        'condition': f'Iteration {i+1}',
        'output_error_signal' : output_error_signal
    }


In [ ]:
plot_loss_landscape_with_state(
    loss_fn=mse_loss, output_vector=y_train,
    states=states_training, show_legend=False, window_size=5, limit_y_axis=False,
)

In [ ]:
num_iterations = len(states_training)
losses = [state['loss'] for state in states_training.values()]
plt.figure(figsize=(5, 3))
plt.plot(range(1, num_iterations + 1), losses, marker='o')
plt.title('Loss History Over Iterations')
plt.xlabel('Iteration')
plt.ylabel('MSE Loss')
plt.yscale('log')

    

Did the loss decrease? How close is the final output to the true function?

## Training the neural network using PyTorch

The code we have written so far is a very basic implementation of a neural network. It is useful for understanding the underlying ideas, but it is neither efficient nor practical for larger problems. In real deep learning applications, we usually rely on dedicated libraries that provide optimized tools for building and training neural networks.

One such library is `PyTorch`, which offers a high-level interface for defining models, computing gradients, and training neural networks efficiently. Let us now use PyTorch to build a similar network and train it to learn the function that we generated at the beginning of this module.

In [ ]:
import torch
import torch.nn as nn

In [ ]:
model = nn.Sequential(
    nn.Linear(1, 3), 
    nn.ReLU(), 
    nn.Linear(3, 1)
)


In [ ]:
# print model weights and biases 
for name, param in model.named_parameters():
    if param.requires_grad:
        # name, value, activation function (if applicable)
        print(f"{name}: {param.data}")
        

Call the PyTorch functions to set the optimiser and the loss function.

In [ ]:
learning_rate_torch = 0.1
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate_torch)

We now need to set up a data preprocessing pipeline. This is important because neural networks generally perform better when the input data is presented on a standardised scale. A common preprocessing step is to subtract the mean of the data and divide by its standard deviation. This process is called **standardisation**.

\begin{equation}
\tag{10}
x_{\text{standardised}} = \frac{x - \mu}{\sigma}
\end{equation}

where $\mu$ is the mean of the data and $\sigma$ is the standard deviation of the data.

In [ ]:
def standardise_data(x):
    mean = 2
    std = 0.1
    standard_data = 2
    return standard_data, mean, std

In [ ]:
# train the model on the same data for comparison
x_train_standard, mean, std = standardise_data(x_sample)
y_train_standard, mean_y, std_y = standardise_data(y_sample)

In [ ]:
# Convert to PyTorch tensors. 
x_train_tensor = torch.tensor(x_train_standard, dtype=torch.float32).unsqueeze(1) # shape (n_samples, 1)
y_train_tensor = torch.tensor(y_train_standard, dtype=torch.float32).unsqueeze(1) # shape (n_samples, 1)


In [ ]:
# Start training runs: 
num_epochs = 1000
loss_history_torch = []
for epoch in range(num_epochs):
    # Set the model to training mode. 
    model.train()
    # Zero the gradients before running the backward pass. 
    optimizer.zero_grad()
    # Forward pass: compute the model output and loss
    outputs = model(x_train_tensor)
    # Compute the loss between the model output and the true labels
    loss = criterion(outputs, y_train_tensor)
    # Backward pass: compute the gradients of the loss with respect to the model parameters
    loss.backward()
    # Update the model parameters using the optimizer
    optimizer.step()
    # Append the current loss to the loss history list 
    loss_history_torch.append(loss.item())

In [ ]:
loss_history_torch = np.array(loss_history_torch)
plt.plot(loss_history_torch)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.yscale("log")

In [ ]:
# Plot the predictions vs true values after training
def predict(model, x, mean_x, std_x, mean_y, std_y):
    # Set the model to evaluation mode. 
    model.eval()
    with torch.no_grad(): # Make predictions without tracking gradients
        # Standardise the input data using the mean and std from training
        x_standard = (x - mean_x) / std_x 
        # Convert the standardised input to a PyTorch tensor 
        x_tensor = torch.tensor(x_standard, dtype=torch.float32).unsqueeze(1)
        # Get the model predictions for the input data and convert to numpy array
        y_pred_standard = model(x_tensor).squeeze().numpy()
        # Unstandardise the predictions using the mean and std from training
        y_pred = y_pred_standard * std_y + mean_y
    return y_pred




In [ ]:
# Let us see how the model performs on the true x values after training
y_pred = predict(model, x_true, mean, std, mean_y, std_y)


In [ ]:
# Plot the predictions vs true values after training
plt.plot(x_true, y_true, color="black", label="True Function", linestyle='--', alpha=0.2)
plt.plot(x_true, y_pred, color="green", label="PyTorch Model Prediction")
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()

Now, increase the number of parameters in the PyTorch model to see how it fits the data. 

In [ ]:
num_params = 64
model_complex = nn.Sequential(
    nn.Linear(1, num_params),
    nn.ReLU(),
    nn.Linear(num_params, 1)
)



In [ ]:
def train_model(model, x_train, y_train, epochs, lr=0.001):
    # define the loss function and optimizer
    criterion = nn.MSELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_history = []
    for epoch in range(epochs):
        # Set the model to training mode.
        
        # Zero the gradients before running the backward pass.
        
        # Forward pass: compute the model output and loss
        
        # Compute the loss between the model output and the true labels
        
        # Backward pass: compute the gradients of the loss with respect to the model parameters
        
        # Update the model parameters using the optimizer
        
        # Append the current loss to the loss history list
        
        
    return model, np.array(loss_history)

In [ ]:
model_complex_trained, loss_history_complex = train_model(\
    model_complex, x_train_tensor, y_train_tensor, epochs=1000, lr=0.01
)

In [ ]:
# Plot the loss history for the complex model
plt.plot(loss_history_complex)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.yscale("log")

In [ ]:
x_test = np.linspace(0, 10, 100)
y_test = degree_3_polynomial(x_test, coeffs=[A, B, C, D], noise_std=0.2)
y_predictions_complex = predict(model_complex_trained, x_test, mean, std, mean_y, std_y)
plt.plot(x_test, y_test, color="black", label="True Function", linestyle='--', alpha=0.2)
plt.plot(x_test, y_predictions_complex, 'm-', label='Predictions (Complex PyTorch Model)')
plt.legend()
plt.xlabel('X')
plt.ylabel('Y')
plt.ylim(-5, 15)


# Universal approximation theorem

- Increasing the number of neurons allows the network to capture more complex features. In the case of ReLU activation functions, this corresponds to fitting **piecewise linear functions** to the data.  

- Increasing the depth of the network further enhances its representational power. A network with a single hidden layer can, in theory, approximate any continuous function, but it may require a very large number of neurons. In contrast, a deeper network can often approximate the same function using fewer neurons by learning hierarchical features.

\begin{equation}
\tag{11}
N_{\text{representation}} \propto N_{\text{neurons}}^{L}
\end{equation}

where $N_{\text{representation}}$ is a measure of the number of functions that can be represented by the network, $N_{\text{neurons}}$ is the number of neurons per layer, and $L$ is the number of layers.

---

However, increasing depth also makes the network harder to train. As gradients are propagated backward through the network, they can become very small due to repeated application of the chain rule. When gradients become too small, the weights in the earlier layers are not updated effectively, and the network struggles to learn. This is known as the **vanishing gradient problem**.

Conversely, if the learning rate is too high, gradients can become excessively large. This can lead to unstable updates, large jumps in the loss landscape, and cause training to diverge. This phenomenon is known as the **exploding gradient problem**.

# Comparing with polynomial regression


In [ ]:
polynomial_degree = 20
num_neurons = 200 

# fit a polynomial for the same data for comparison
polynomial_coefficients = np.polyfit(x_sample, y_sample, polynomial_degree)
y_predictions_polynomial = np.polyval(polynomial_coefficients, x_test)

# fit a neural network with more neurons for the same data for comparison
model_more_neurons = nn.Sequential(
    nn.Linear(1, num_neurons),
    nn.ReLU(),
    nn.Linear(num_neurons, 1)
)
model_more_neurons_trained, loss_history_more_neurons = train_model(
    model_more_neurons, x_train_tensor, y_train_tensor, epochs=1000, lr=0.01
)
y_predictions_more_neurons = predict(model_more_neurons_trained, x_test, mean, std, mean_y, std_y)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].plot(x_test, y_test, color="black", label="True Function", linestyle='--')
ax[0].scatter(x_sample, y_sample, color="red", label="Sampled Data")
ax[0].plot(x_test, y_predictions_polynomial, 'c-', label=f'Predictions (Polynomial Degree {polynomial_degree})')
ax[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')
ax[0].set_ylim(-5, 15)
ax[0].set_xlabel('X')
ax[0].set_ylabel('Y')
ax[1].plot(x_test, y_test, color="black", label="True Function", linestyle='--')
ax[1].scatter(x_sample, y_sample, color="red", label="Sampled Data")
ax[1].plot(x_test, y_predictions_more_neurons, 'y-', label=f'Predictions (NN with {num_neurons} Neurons)')
ax[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')
ax[1].set_ylim(-5, 15)
ax[1].set_xlabel('X')
ax[1].set_ylabel('Y')

fig.tight_layout()
